# S4 · Проверка модели на новых сообщениях

Исторический реальный корпус UCI SMS Spam Collection, Almeida и Hidalgo, [DOI 10.24432/C5CC84](https://doi.org/10.24432/C5CC84), CC BY 4.0. Файл уже находится в `data/`, сеть не нужна.

Основной маршрут: 75 минут. Объект — одно сообщение. Повторы удаляются до разбиения. Словарь free, win, prize задан заранее. Параметры обучаются на обучающей части, порог выбирается на валидации. Блок 6 с тестом выполняется после фиксации решения. Условная цена заранее принята равной 5 × FP + FN.

Отправители и кампании неизвестны, похожие шаблоны могут оставаться. Этот опыт оценивает работу на отложенной части исторического корпуса.

### Подготовлено: импорты и чтение файла

In [1]:
from pathlib import Path

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

data_path = Path("data/sms-spam.tsv")
if not data_path.exists():
    data_path = Path("../data/sms-spam.tsv")
if not data_path.exists():
    data_path = Path("../../data/sms-spam.tsv")
raw_rows = []
for line in data_path.read_text(encoding="utf-8").splitlines():
    label, message = line.split("\t", 1)
    raw_rows.append((label, message))
print("Строк в исходном файле:", len(raw_rows))


Строк в исходном файле: 5574


## 1. Проверка данных и удаление повторов

`seen` хранит уже встретившиеся нормализованные тексты. `continue` переходит к следующей строке. После очистки: 5159 сообщений, из них 642 спам.

In [2]:
messages = []
labels = []
seen = set()
for label, message in raw_rows:
    assert label in ["ham", "spam"]
    key = " ".join(message.lower().split())
    assert key != ""
    if key in seen:
        continue
    seen.add(key)
    messages.append(message)
    labels.append(int(label == "spam"))

print("После удаления повторов:", len(messages))
print("Спам:", sum(labels))
print("Обычные:", len(labels) - sum(labels))


После удаления повторов: 5159
Спам: 642
Обычные: 4517


### Подготовлено: тот же признак, что на S3

In [3]:
def count_words(message):
    count = 0
    for word in message.lower().split():
        word = word.strip(".,!?;:")
        if word in ["free", "win", "prize"]:
            count = count + 1
    return count


## 2. Таблица признаков и три части данных

В X одна строка содержит список из одного признака. `ids` — номера строк. `stratify` сохраняет близкие доли классов, `random_state` фиксирует разбиение. Размеры: 3095 / 1032 / 1032.

In [4]:
X = []
for message in messages:
    X.append([count_words(message)])
X = np.array(X)
y = np.array(labels)
ids = np.arange(len(y))

train_ids, other_ids = train_test_split(ids, test_size=0.4, random_state=25, stratify=y)
validation_ids, test_ids = train_test_split(
    other_ids, test_size=0.5, random_state=25, stratify=y[other_ids]
)
print("Форма X:", X.shape, "Форма y:", y.shape)
print("Обучение / валидация / тест:", len(train_ids), len(validation_ids), len(test_ids))


Форма X: (5159, 1) Форма y: (5159,)
Обучение / валидация / тест: 3095 1032 1032


## 3. Обучение библиотечной модели

`C=inf` отключает регуляризацию, чтобы сохранить функцию потерь из L1. Используется библиотечный оптимизатор L-BFGS. `max_iter` — предел его итераций. Две колонки predict_proba соответствуют classes_=[0,1]; [:,1] берёт вторую.

In [5]:
model = LogisticRegression(C=float("inf"), max_iter=1000)
model.fit(X[train_ids], y[train_ids])
validation_probability = model.predict_proba(X[validation_ids])[:, 1]
print("Порядок классов:", model.classes_)
print("w =", round(model.coef_[0, 0], 3), "b =", round(model.intercept_[0], 3))
print("Первые пять оценок:", np.round(validation_probability[:5], 3))


Порядок классов: [0 1]
w = 3.433 b = -2.386
Первые пять оценок: [0.084 0.74  0.084 0.084 0.084]


## 4. Два вида ошибок и точка отсчёта

Считаем случаи по одному. TN — обычное во входящих; FP — обычное в спаме; FN — спам во входящих; TP — спам в спаме. Сумма равна числу сообщений.

In [6]:
def count_errors(actual, probability, threshold):
    fp = 0
    fn = 0
    tp = 0
    tn = 0
    for i in range(len(actual)):
        predicted = int(probability[i] >= threshold)
        if actual[i] == 0 and predicted == 1:
            fp = fp + 1
        elif actual[i] == 1 and predicted == 0:
            fn = fn + 1
        elif actual[i] == 1 and predicted == 1:
            tp = tp + 1
        else:
            tn = tn + 1
    return tn, fp, fn, tp


always_zero = np.zeros(len(validation_ids))
baseline = count_errors(y[validation_ids], always_zero, 0.5)
print("Всегда обычное, TN FP FN TP:", baseline)
print("Модель, порог 0.5:", count_errors(y[validation_ids], validation_probability, 0.5))


Всегда обычное, TN FP FN TP: (903, 0, 129, 0)
Модель, порог 0.5: (889, 14, 73, 56)


## 5. Выбор порога на валидации

Выполняем только четыре заранее заданных сравнения на валидации. При одинаковой цене сохраняется первый порог. Фиксируем 0.75.

In [7]:
best_cost = float("inf")
best_threshold = None
validation_results = []
for threshold in [0.10, 0.25, 0.50, 0.75]:
    tn, fp, fn, tp = count_errors(y[validation_ids], validation_probability, threshold)
    cost = 5 * fp + fn
    validation_results.append([threshold, tn, fp, fn, tp, cost])
    print("Порог:", threshold, "FP:", fp, "FN:", fn, "Цена:", cost)
    if cost < best_cost:
        best_cost = cost
        best_threshold = threshold

print("Фиксируем порог:", best_threshold)


Порог: 0.1 FP: 14 FN: 73 Цена: 143
Порог: 0.25 FP: 14 FN: 73 Цена: 143
Порог: 0.5 FP: 14 FN: 73 Цена: 143
Порог: 0.75 FP: 1 FN: 114 Цена: 119
Фиксируем порог: 0.75


### Подготовлено: реальные ошибки только на валидации

In [8]:
for error_name, true_label, predicted_label in [("FP", 0, 1), ("FN", 1, 0)]:
    for i in range(len(validation_ids)):
        row_id = validation_ids[i]
        prediction = int(validation_probability[i] >= best_threshold)
        if y[row_id] == true_label and prediction == predicted_label:
            print(error_name, "x =", X[row_id, 0], "p =", round(validation_probability[i], 3))
            print(messages[row_id])
            break


FP x = 2 p = 0.989
Fighting with the world is easy, u either win or lose bt fightng with some1 who is close to u is dificult if u lose - u lose if u win - u still lose.
FN x = 1 p = 0.74
Get ur 1st RINGTONE FREE NOW! Reply to this msg with TONE. Gr8 TOP 20 tones to your phone every week just £1.50 per wk 2 opt out send STOP 08452810071 16


## 6. Единственная итоговая проверка

Все решения приняты до этого блока. На этой версии данных знаменатели precision и recall положительны. В общем случае при нулевом знаменателе доля не определена и требует явной договорённости. Результат: 9 верных срабатываний из 9, найдено 9 из 128 сообщений спама. Изменения после просмотра теста требуют новой независимой проверки.

In [9]:
test_probability = model.predict_proba(X[test_ids])[:, 1]
tn, fp, fn, tp = count_errors(y[test_ids], test_probability, best_threshold)
accuracy = (tp + tn) / len(test_ids)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
test_cost = 5 * fp + fn
test_baseline_cost = int(sum(y[test_ids]))
print("Тест: TN FP FN TP =", tn, fp, fn, tp)
print("Accuracy:", round(accuracy, 3))
print("Precision:", round(precision, 3), "Recall:", round(recall, 3))
print("Цена модели / постоянного ответа:", test_cost, test_baseline_cost)


Тест: TN FP FN TP = 904 0 119 9
Accuracy: 0.885
Precision: 1.0 Recall: 0.07
Цена модели / постоянного ответа: 119 128
